# MNIST classification using the OpenEye accelerator (Pytorch version)

This notebook demonstrates how to perform MNIST digit classification 
using a PyTorch model optimized for the OpenEye accelerator. 
It covers loading the dataset, defining the model architecture, 
training the model, deploying it on the OpenEye accelerator, and 
evaluating its performance.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch Version: {torch.__version__}")


In [ ]:
""" Define the neural network architecture.

In this tutorial, we use a host device for training and the OpenEye accelerator
purely for inference. Hence, we define a simple feedforward neural network and 
train it on the host device with the available hardware acceleration. 

PyTorch can run on CPU, GPU or MPS. Here, we check for GPU or MPS availability.
"""
device = torch.device("cuda" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

print(f"Using device: {device}")

In [ ]:
# Hyperparameters are settings that influence training
BATCH_SIZE = 64          # Number of images per training step
LEARNING_RATE = 0.001    # Step size when learning (too large = unstable, too small = slow)
EPOCHS = 5               # How many times to iterate through entire dataset
INPUT_SIZE = 28 * 28     # MNIST images are 28x28 pixels = 784 pixels
HIDDEN_SIZE = 128        # Number of neurons in hidden layer
NUM_CLASSES = 10         # Digits 0-9 = 10 classes

In [ ]:
# Transform: Converts images to PyTorch tensors and normalizes them
# Normalization: (value - mean) / standard deviation
# This helps the neural network learn better
transform = transforms.Compose([
    transforms.ToTensor(),                    # Image to tensor (0-255 -> 0-1)
    transforms.Normalize((0.1307,), (0.3081,))  # Normalization with MNIST statistics
])

In [ ]:
# Download datasets (first time only) and load
train_dataset = datasets.MNIST(
    root='./data',           # Storage location
    train=True,              # Training data
    download=True,           # Download if not present
    transform=transform      # Apply transformations
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,             # Test data
    download=True,
    transform=transform
)

In [ ]:
# DataLoader: Loads data in batches and shuffles them
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,            # Shuffle data for each epoch
    num_workers=2            # Parallel workers for loading
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
print(f"Training data: {len(train_dataset)} images")
print(f"Test data: {len(test_dataset)} images")
print(f"Number of batches per epoch: {len(train_loader)}")

In [ ]:
from mnist_conv_net import SimpleMNISTConvNet



In [ ]:
"""Create model and move to device"""
model = SimpleMNISTConvNet().to(device)  
print("\nModel Architecture:")
print(model)

# Count number of parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


In [ ]:
# Loss function: Measures how wrong the predictions are
# CrossEntropyLoss combines LogSoftmax and NLLLoss
# Perfect for classification problems!
criterion = nn.CrossEntropyLoss()

# Optimizer: Updates the network's weights
# Adam is an improved version of Stochastic Gradient Descent
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Optional: Learning Rate Scheduler (reduces learning rate over time)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.7)

In [ ]:
def train(model, device, train_loader, criterion, optimizer, epoch):
    """
    Trains the model for one epoch
    
    Args:
        model: The neural network
        device: CPU or CUDA
        train_loader: DataLoader with training data
        criterion: Loss function
        optimizer: Optimizer
        epoch: Current epoch number
    
    Returns:
        Average loss
    """
    model.train()  # Sets model to training mode (important for Dropout, BatchNorm)
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        # Move data to device (CPU or GPU)
        data, target = data.to(device), target.to(device)
        
        # 1. Zero the gradients (important!)
        optimizer.zero_grad()
        
        # 2. Forward Pass: Compute predictions
        output = model(data)
        
        # 3. Calculate loss
        loss = criterion(output, target)
        
        # 4. Backward Pass: Compute gradients
        loss.backward()
        
        # 5. Optimizer Step: Update weights
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        
        # Show progress every 100 batches
        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\t'
                  f'Loss: {loss.item():.6f}')
    
    avg_loss = running_loss / len(train_loader)
    accuracy = 100. * correct / total
    
    print(f'\nTraining: Average Loss: {avg_loss:.4f}, '
          f'Accuracy: {correct}/{total} ({accuracy:.2f}%)\n')
    
    return avg_loss

In [ ]:
def test(model, device, test_loader, criterion):
    """
    Evaluates the model on test data
    
    Args:
        model: The neural network
        device: CPU or CUDA
        test_loader: DataLoader with test data
        criterion: Loss function
    
    Returns:
        Average loss and accuracy
    """
    model.eval()  # Sets model to evaluation mode (disables Dropout)
    
    test_loss = 0
    correct = 0
    
    # torch.no_grad() disables gradient computation (saves memory and time)
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            # Forward Pass
            output = model(data)
            
            # Sum up loss
            test_loss += criterion(output, target).item()
            
            # Prediction is the class with highest probability
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    
    test_loss /= len(test_loader)
    accuracy = 100. * correct / len(test_loader.dataset)
    
    print(f'Test: Average Loss: {test_loss:.4f}, '
          f'Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')
    
    return test_loss, accuracy

In [ ]:
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70 + "\n")

train_losses = []
test_losses = []
test_accuracies = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch}/{EPOCHS}")
    print(f"{'='*70}")
    
    # Train
    train_loss = train(model, device, train_loader, criterion, optimizer, epoch)
    train_losses.append(train_loss)
    
    # Test
    test_loss, test_acc = test(model, device, test_loader, criterion)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    
    # Adjust learning rate
    scheduler.step()
    print(f"Learning Rate: {scheduler.get_last_lr()[0]:.6f}")

print("\n" + "="*70)
print("TRAINING COMPLETED!")
print("="*70)
print(f"Best Test Accuracy: {max(test_accuracies):.2f}%")

In [ ]:

def visualize_predictions(model, device, test_loader, num_images=10):
    """Shows examples with predictions"""
    model.eval()
    
    # Get one batch
    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    
    images = images.to(device)
    with torch.no_grad():
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
    
    # Back to CPU for Matplotlib
    images = images.cpu()
    predictions = predictions.cpu()
    
    # Plot
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.ravel()
    
    for idx in range(num_images):
        img = images[idx].squeeze()
        true_label = labels[idx].item()
        pred_label = predictions[idx].item()
        
        axes[idx].imshow(img, cmap='gray')
        color = 'green' if true_label == pred_label else 'red'
        axes[idx].set_title(f'True: {true_label}\nPred: {pred_label}', 
                           color=color, fontweight='bold', fontsize=11)
        axes[idx].axis('off')
    
    plt.tight_layout()
    print("Prediction examples saved: predictions.png")


# Create visualizations
visualize_predictions(model, device, test_loader)

In [ ]:
# Save model completely, so that it can be reloaded later without knowing architecture
model_path = 'mnist_unquantized_model.pth'
torch.save({
    'epoch': EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'test_accuracy': max(test_accuracies),
}, model_path)
print(f"\nModel saved: {model_path}")

In [ ]:
def predict_single_image(model, image_tensor, device):
    """
    Makes a prediction for a single image
    
    Args:
        model: Trained model
        image_tensor: Image as tensor (1, 28, 28)
        device: CPU or CUDA
    
    Returns:
        Predicted class and probabilities
    """
    model.eval()
    
    # Add batch dimension: (1, 28, 28) -> (1, 1, 28, 28)
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)
    
    image_tensor = image_tensor.to(device)
    
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = F.softmax(output, dim=1)
        predicted_class = output.argmax(dim=1).item()
        confidence = probabilities[0][predicted_class].item()
    
    return predicted_class, confidence, probabilities.cpu().numpy()[0]


# Example: Prediction for first test image
test_image, test_label = test_dataset[0]
predicted, confidence, probs = predict_single_image(model, test_image, device)

In [ ]:
import torchao
print(torchao.__version__)  # Print torchao version

In [ ]:
"""Before we can map the model to the OpenEye accelerator, we need to quantize
it to 8-bit integer arithmetic.

Hence, we define a function to quantize the trained model using PyTorch's static
quantization. The quantization performed here will convert the floating-point
weights and activations of the model to 8-bit integers in some simple "vanilla"
manner. Other more advanced quantization techniques may yield better accuracy,
but are beyond the scope of this demo. 
"""

def quantize_model(model, test_loader, device):
    """
    Quantizes a PyTorch model to 8-bit integer arithmetic
    Args:
        model: Trained PyTorch model
        test_loader: DataLoader for calibration
        device: CPU or CUDA
    """
    import copy
    # take a few examples from the test set for calibration
    calibration_batches = 10
    calibration_data = []
    for i, (data, target) in enumerate(test_loader):
        if i >= calibration_batches:
            break
        calibration_data.append(data)

    non_quant_model = copy.deepcopy(model)
    non_quant_model.to('cpu')  # Quantization typically runs on CPU
    non_quant_model.eval()

    # torch.export.export needs a single example input that matches the model's forward signature
    # We use the first batch as the example input with dynamic batch size
    example_input = (calibration_data[0],)
    
    # Define dynamic shapes to allow variable batch size
    dynamic_shapes = {"x": {0: torch.export.Dim("batch", min=1, max=1024)}}

    m = torch.export.export(non_quant_model, example_input, dynamic_shapes=dynamic_shapes).module()

    from torchao.quantization.pt2e.quantize_pt2e import (
        prepare_pt2e,
        convert_pt2e)

    from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
        get_symmetric_quantization_config,
        XNNPACKQuantizer)
    
    quantizer = XNNPACKQuantizer().set_global(get_symmetric_quantization_config())
    m = prepare_pt2e(m, quantizer)

    # Run calibration with the collected data
    with torch.no_grad():
        for data in calibration_data:
            m(data)

    m = convert_pt2e(m)

    return m

quantized_model = quantize_model(model, test_loader, device)

quantized_model.conv1

quantized_model.fc1


In [ ]:
"""Just for info, we define a function to compare the file sizes of the original
float32 model vs the quantized int8 model."""

def compare_model_sizes(original_model, quantized_model):
    """
    Compare file sizes of original vs quantized model
    
    Args:
        original_model: Original float32 model
        quantized_model: Quantized int8 model
    """
    import os
    
    # Save both models temporarily
    torch.save(original_model.state_dict(), 'original_model.pth')
    torch.save(quantized_model.state_dict(), 'quantized_model.pth')
    
    # Get file sizes
    original_size = os.path.getsize('original_model.pth')
    quantized_size = os.path.getsize('quantized_model.pth')
    
    print("\n" + "="*70)
    print("MODEL SIZE COMPARISON")
    print("="*70)
    print(f"Original model (float32):  {original_size / 1024:.2f} KB")
    print(f"Quantized model (int8):    {quantized_size / 1024:.2f} KB")
    print(f"Size reduction:            {(1 - quantized_size/original_size)*100:.1f}%")
    print(f"Compression ratio:         {original_size/quantized_size:.2f}x")
    
    # Cleanup
    os.remove('original_model.pth')
    os.remove('quantized_model.pth')


In [ ]:
def benchmark_inference_speed(original_model, quantized_model, test_loader, device):
    """
    Compare inference speed of original vs quantized model
    
    Args:
        original_model: Original float32 model
        quantized_model: Quantized int8 model (exported model)
        test_loader: DataLoader with test data
        device: CPU or CUDA
    """
    import time
    from torchao.quantization.pt2e import allow_exported_model_train_eval
    
    print("\n" + "="*70)
    print("INFERENCE SPEED COMPARISON")
    print("="*70)
    
    # Get one batch for testing
    data_iter = iter(test_loader)
    test_data, _ = next(data_iter)
    
    # Benchmark original model
    original_model_cpu = original_model.cpu()
    original_model_cpu.eval()
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = original_model_cpu(test_data)
    
    # Measure
    num_iterations = 100
    start_time = time.time()
    with torch.no_grad():
        for _ in range(num_iterations):
            _ = original_model_cpu(test_data)
    original_time = (time.time() - start_time) / num_iterations * 1000  # ms
    
    # Benchmark quantized model
    # Allow eval() for exported models
    allow_exported_model_train_eval(quantized_model)
    quantized_model.eval()
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = quantized_model(test_data)
    
    # Measure
    start_time = time.time()
    with torch.no_grad():
        for _ in range(num_iterations):
            _ = quantized_model(test_data)
    quantized_time = (time.time() - start_time) / num_iterations * 1000  # ms
    
    print(f"Original model (float32):  {original_time:.3f} ms per batch")
    print(f"Quantized model (int8):    {quantized_time:.3f} ms per batch")
    print(f"Speedup:                   {original_time/quantized_time:.2f}x faster")
    print(f"Time saved:                {((original_time-quantized_time)/original_time)*100:.1f}%")


In [ ]:
def evaluate_quantized_model(quantized_model, test_loader):
    """
    Evaluate accuracy of quantized model
    
    Args:
        quantized_model: Quantized model to evaluate (exported model)
        test_loader: DataLoader with test data
    
    Returns:
        Accuracy percentage
    """
    from torchao.quantization.pt2e import allow_exported_model_train_eval
    
    # Allow eval() for exported models
    allow_exported_model_train_eval(quantized_model)
    quantized_model.eval()
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            output = quantized_model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
    
    accuracy = 100. * correct / total
    return accuracy


In [ ]:
# Execute quantization
print("\nStarting quantization process...")
quantized_model = quantize_model(model, test_loader, device)

# Compare model sizes
compare_model_sizes(model, quantized_model)

# Benchmark inference speed
benchmark_inference_speed(model, quantized_model, test_loader, device)

# Evaluate quantized model accuracy
print("\n" + "="*70)
print("ACCURACY COMPARISON")
print("="*70)
print(f"Original model accuracy:   {max(test_accuracies):.2f}%")

quantized_accuracy = evaluate_quantized_model(quantized_model, test_loader)
print(f"Quantized model accuracy:  {quantized_accuracy:.2f}%")
print(f"Accuracy difference:       {abs(max(test_accuracies) - quantized_accuracy):.2f}%")

if quantized_accuracy >= max(test_accuracies) - 1.0:
    print("✅ Excellent! Accuracy loss is less than 1%")
elif quantized_accuracy >= max(test_accuracies) - 2.0:
    print("✅ Good! Accuracy loss is acceptable (1-2%)")
else:
    print("⚠️  Consider using Quantization Aware Training for better accuracy")


# Save quantized model
quantized_model_path = 'mnist_model_quantized.pth'
torch.save(quantized_model.state_dict(), quantized_model_path)
print(f"\nQuantized model saved: {quantized_model_path}")



# Step 8: Export Quantized Model for OpenEye

Now we need to export the quantized model to a format that can be used with the OpenEye accelerator. We'll save it as an ONNX model which the OpenEye infrastructure can load.

In [ ]:
"""Export the quantized model to ONNX format for OpenEye compatibility"""
import os

# Create output directory for all generated files
output_dir = 'mnist_pytorch_cocotb'
os.makedirs(output_dir, exist_ok=True)
print(f"Created output directory: {output_dir}/")

# First, we need to export the quantized model to ONNX
# Get a sample input for tracing
sample_input = next(iter(test_loader))[0][:1]  # Get one sample from test set

# Export to ONNX
onnx_model_path = os.path.join(output_dir, 'mnist_model_quantized.onnx')

try:
    # For the quantized model, we need to use torch.onnx.export
    torch.onnx.export(
        quantized_model,              # model being run
        sample_input,                  # model input (or a tuple for multiple inputs)
        onnx_model_path,              # where to save the model
        export_params=True,            # store the trained parameter weights inside the model file
        opset_version=13,              # the ONNX version to export the model to
        do_constant_folding=True,      # whether to execute constant folding for optimization
        input_names=['input'],         # the model's input names
        output_names=['output'],       # the model's output names
        dynamic_axes={
            'input': {0: 'batch_size'},    # variable length axes
            'output': {0: 'batch_size'}
        }
    )
    print(f"✅ Quantized model exported to ONNX: {onnx_model_path}")
    
    # Check file size
    onnx_size_kb = os.path.getsize(onnx_model_path) / 1024
    print(f"   ONNX model size: {onnx_size_kb:.2f} KB")
    
except Exception as e:
    print(f"⚠️  ONNX export failed: {e}")
    print("   Continuing with alternative approach...")
    
    # Alternative: Save as PyTorch traced model
    traced_model_path = os.path.join(output_dir, 'mnist_model_quantized_traced.pt')
    traced_model = torch.jit.trace(quantized_model, sample_input)
    torch.jit.save(traced_model, traced_model_path)
    print(f"✅ Saved traced PyTorch model: {traced_model_path}")


# Step 9: Load Model with OpenEye Model Loader

Now we'll use OpenEye's unified model loader to load our quantized model and prepare it for hardware simulation.

In [ ]:
"""Load the model using OpenEye's unified model loader"""
import sys
import os

# Add OpenEye to path
openeye_base = os.path.abspath(os.path.join(os.pardir, os.pardir))
sys.path.insert(0, os.path.join(openeye_base, "src"))

from open_eye.model_loader import load_model
from open_eye.layer_adapter import adapt_layers_for_keras

# Try to load the ONNX model, fallback to traced PyTorch if needed
try:
    if os.path.exists(onnx_model_path):
        print(f"Loading model from {onnx_model_path}...")
        openeye_model = load_model(onnx_model_path, format='onnx')
        model_source = "ONNX"
    elif os.path.exists('mnist_model_quantized_traced.pt'):
        print(f"Loading model from mnist_model_quantized_traced.pt...")
        openeye_model = load_model('mnist_model_quantized_traced.pt', format='pytorch')
        model_source = "PyTorch (traced)"
    else:
        print("⚠️  No exported model found. Please run the export cell first.")
        raise FileNotFoundError("Model file not found")
        
    print("\n" + "="*70)
    print(f"OpenEye Model Loaded from {model_source}")
    print("="*70)
    print(f"Framework:    {openeye_model.framework}")
    print(f"Layers:       {openeye_model.num_layers}")
    print(f"Input shape:  {openeye_model.input_shape}")
    print(f"Output shape: {openeye_model.output_shape}")
    print(f"Quantized:    {openeye_model.is_quantized}")
    
    print("\nLayer Structure:")
    for i, layer in enumerate(openeye_model.layers):
        layer_name = layer.__class__.__name__
        in_shape = layer.input_shape if hasattr(layer, 'input_shape') else 'N/A'
        out_shape = layer.output_shape if hasattr(layer, 'output_shape') else 'N/A'
        print(f"  {i}: {layer_name:30s} {str(in_shape):20s} → {str(out_shape)}")
    
    print("\n✅ Model loaded successfully!")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    import traceback
    traceback.print_exc()


# Step 10: Adapt Layers for OpenEye Hardware

We'll adapt the layers to make them compatible with OpenEye's hardware simulation infrastructure.

In [ ]:
"""Adapt layers for Keras compatibility and inspect details"""

# Adapt layers for OpenEye hardware
adapted_layers = adapt_layers_for_keras(openeye_model.layers)

print("Adapted Layers for OpenEye Hardware:")
print("=" * 70)

for i, layer in enumerate(adapted_layers):
    layer_name = layer.name if hasattr(layer, 'name') else 'Unknown'
    output_shape = layer.output.shape if hasattr(layer, 'output') else 'N/A'
    
    print(f"\nLayer {i}: {layer_name}")
    print(f"  Input shape:  {layer.input.shape if hasattr(layer, 'input') else 'N/A'}")
    print(f"  Output shape: {output_shape}")
    
    # Conv2D details
    if layer_name == 'conv2d':
        print(f"  Kernel size:  {layer.kernel_size if hasattr(layer, 'kernel_size') else 'N/A'}")
        print(f"  Filters:      {layer.filters if hasattr(layer, 'filters') else 'N/A'}")
        print(f"  Strides:      {layer.strides if hasattr(layer, 'strides') else 'N/A'}")
        print(f"  Padding:      {layer.padding if hasattr(layer, 'padding') else 'N/A'}")
        if hasattr(layer, 'kernel'):
            print(f"  Weight shape: {layer.kernel.shape}")
            weights = layer.get_weights()
            if len(weights) > 0:
                print(f"  Num weights:  {np.prod(weights[0].shape):,}")
            if len(weights) > 1:
                print(f"  Num biases:   {len(weights[1]):,}")
    
    # Dense details
    elif layer_name == 'dense':
        print(f"  Units:        {layer.units if hasattr(layer, 'units') else 'N/A'}")
        if hasattr(layer, 'kernel'):
            print(f"  Weight shape: {layer.kernel.shape}")
            weights = layer.get_weights()
            if len(weights) > 0:
                print(f"  Num weights:  {np.prod(weights[0].shape):,}")
            if len(weights) > 1:
                print(f"  Num biases:   {len(weights[1]):,}")
    
    # Flatten details
    elif layer_name == 'flatten':
        print(f"  (No parameters)")
    
    # MaxPooling details
    elif layer_name == 'max_pooling2d':
        if hasattr(layer, 'pool_size'):
            print(f"  Pool size:    {layer.pool_size}")
            print(f"  Strides:      {layer.strides if hasattr(layer, 'strides') else 'N/A'}")

print("\n✅ All layers adapted successfully!")


# Step 11: Prepare Test Data for Cocotb Simulation

We'll prepare some test images from the MNIST dataset to use in the cocotb simulation.

In [ ]:
"""Prepare test data for cocotb simulation"""
from torchao.quantization.pt2e import allow_exported_model_train_eval

# Select a few test samples for simulation
num_test_samples = 10
test_samples = []
test_labels = []

for i in range(num_test_samples):
    img, label = test_dataset[i]
    test_samples.append(img)
    test_labels.append(label)

test_samples = torch.stack(test_samples)
test_labels_array = np.array(test_labels)

print(f"Prepared {num_test_samples} test samples for simulation")
print(f"Test samples shape: {test_samples.shape}")
print(f"Test labels: {test_labels_array}")

# Get predictions from the quantized model for comparison
# Allow eval() for exported models
allow_exported_model_train_eval(quantized_model)
quantized_model.eval()

with torch.no_grad():
    quantized_predictions = quantized_model(test_samples)
    quantized_pred_labels = quantized_predictions.argmax(dim=1).numpy()

print(f"\nQuantized model predictions: {quantized_pred_labels}")
print(f"True labels:                 {test_labels_array}")
print(f"Accuracy on test samples:    {(quantized_pred_labels == test_labels_array).sum() / num_test_samples * 100:.1f}%")

# Save test data for cocotb in the output directory
test_data_dir = os.path.join(output_dir, 'test_data')
os.makedirs(test_data_dir, exist_ok=True)

# Save test images as numpy arrays
for i in range(num_test_samples):
    img_array = test_samples[i].numpy()
    np.save(os.path.join(test_data_dir, f'test_image_{i}.npy'), img_array)

# Save labels
np.save(os.path.join(test_data_dir, 'test_labels.npy'), test_labels_array)
np.save(os.path.join(test_data_dir, 'quantized_predictions.npy'), quantized_pred_labels)

print(f"\n✅ Test data saved to {test_data_dir}/")


# Step 12: Create Cocotb Test Script

Now we'll create a cocotb test script that will simulate the model on the OpenEye hardware accelerator.

In [ ]:
"""Create cocotb test script for MNIST PyTorch model"""

cocotb_test_script = '''#!/usr/bin/env python3
# This file is part of the OpenEye project.
# All rights reserved. © Fachhochschule Dortmund - University of Applied Sciences and Arts.
# SPDX-License-Identifier: SHL-2.1

"""Cocotb test for MNIST PyTorch quantized model on OpenEye accelerator"""

import os
import sys
import cocotb
import cocotb_test.simulator
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge
import pytest
import numpy as np
import logging

# Add OpenEye to path
tests_dir = os.path.abspath(os.path.dirname(__file__))
openeye_base = os.path.abspath(os.path.join(tests_dir, os.pardir, os.pardir))
sys.path.insert(0, os.path.join(openeye_base, "src"))
sys.path.insert(0, os.path.join(openeye_base, "test"))

import open_eye.test_utils_main as test_utils_main

logger = logging.getLogger("cocotb")

# Clock configuration
clk_cycle = 20
clk_cycle_unit = "ns"
clk_delay_in = 100
clk_delay_unit_in = "ps"
clk_delay_out = 100
clk_delay_unit_out = "ps"

@pytest.mark.parametrize("test_idx", [0])
def test_mnist_pytorch(test_idx):
    """Run cocotb simulation for MNIST PyTorch model"""
    dut = 'OpenEye_Parallel'
    module = 'mnist_pytorch_tb'
    toplevel = dut
    
    hdl_dir = os.path.abspath(os.path.join(openeye_base, "hdl"))
    verilog_sources = test_utils_main.get_verilog_sources(hdl_dir)
    
    target_dir = os.path.join(tests_dir, '.temp', 'mnist_pytorch_test')
    
    results = cocotb_test.simulator.run(
        python_search=[tests_dir],
        verilog_sources=verilog_sources,
        toplevel=toplevel,
        module=module,
        sim_build=target_dir,
        testcase='mnist_pytorch_test',
        force_compile=True,
        simulator="icarus",
        extra_env={
            "CLOCK_LEN": str(clk_cycle),
            "CLOCK_UNIT": clk_cycle_unit,
            "CLOCK_DELAY_INPUT": str(clk_delay_in),
            "CLOCK_DELAY_UNIT_INPUT": clk_delay_unit_in,
            "CLOCK_DELAY_OUTPUT": str(clk_delay_out),
            "CLOCK_DELAY_UNIT_OUTPUT": clk_delay_unit_out,
            "MODEL_PATH": os.path.abspath("mnist_model_quantized.onnx"),
            "TEST_DATA_DIR": os.path.abspath("test_data")
        }
    )

if __name__ == '__main__':
    test_mnist_pytorch(0)
'''

# Write the cocotb test script to the output directory
cocotb_test_file = os.path.join(output_dir, 'test_mnist_pytorch.py')
with open(cocotb_test_file, 'w') as f:
    f.write(cocotb_test_script)

print(f"✅ Created cocotb test script: {cocotb_test_file}")


In [ ]:
"""Create cocotb testbench and required Verilog header files for MNIST PyTorch model

This cell creates the necessary Verilog header files that define hardware parameters
and register map layouts for the OpenEye accelerator.
"""

# ============================================================================
# 1. HARDWARE PARAMETERS (parameters.vh)
# ============================================================================
# These define the physical configuration of the OpenEye accelerator hardware:
# - Processing Element (PE) array dimensions
# - Global buffer sizes for activations, weights, and partial sums

parameters_vh_content = '''// OpenEye Hardware Parameters for MNIST PyTorch Model
// Auto-generated by mnist_pytorch.ipynb

// PE Array Configuration
parameter CLUSTER_ROWS  = 4,      // Number of PE cluster rows
parameter PE_ROWS       = 3,      // Number of PE rows per cluster
parameter PE_COLUMNS    = 4,      // Number of PE columns

// Global Buffer Sizes
parameter NUM_GLB_IACT  = 6,      // Global buffer entries for input activations
parameter NUM_GLB_WGHT  = 3,      // Global buffer entries for weights
parameter NUM_GLB_PSUM  = 4,      // Global buffer entries for partial sums
'''

parameters_vh_file = os.path.join(output_dir, 'parameters.vh')
with open(parameters_vh_file, 'w') as f:
    f.write(parameters_vh_content)

print(f"✅ Created parameters.vh: Hardware configuration")
print(f"   - PE Array: {4}×{3} = 12 PEs per cluster, {4} clusters")
print(f"   - Global Buffers: {6} iact, {3} wght, {4} psum entries")

# ============================================================================
# 2. REGISTER MAP PARAMETERS (regmap_params.vh)
# ============================================================================
# The register map defines how layer configuration parameters are packed into
# 64-bit DMA words and transmitted to the hardware. Each "transmission" contains
# multiple parameters packed together to minimize data transfer overhead.
#
# CONCEPT:
# --------
# The OpenEye accelerator needs ~40 different configuration parameters per layer
# (kernel size, stride, input dimensions, etc.). These are packed into 64-bit
# DMA words and sent to the hardware in 4 separate transmissions.
#
# PARAMETER_POS_X_Y defines the BIT POSITION of parameter Y in transmission X.
# For example:
#   PARAMETER_POS_0_0 = 0          → First parameter starts at bit 0
#   PARAMETER_POS_0_1 = 8          → Second parameter starts at bit 8 (after 8-bit first param)
#   PARAMETER_POS_0_2 = 11         → Third parameter starts at bit 11 (after 3-bit second param)
#
# The Verilog code uses these positions to extract parameters from the DMA word:
#   stride_x_reg <= dma_data_i[PARAMETER_POS_0_1 + 2 : PARAMETER_POS_0_1];
#   This extracts 3 bits starting at position PARAMETER_POS_0_1 (bits 10:8)
#
# WHY MULTIPLE TRANSMISSIONS?
# ----------------------------
# A 64-bit DMA word can only hold so many parameters. For complex layers,
# we need multiple DMA transfers (transmissions 0-3) to configure all settings.

regmap_params_vh_content = '''`ifndef REGMAP_PARAMS_VH
`define REGMAP_PARAMS_VH

// Auto-generated register map parameters for MNIST PyTorch Model
// Generated by mnist_pytorch.ipynb
//
// PURPOSE: Define bit positions for layer configuration parameters within
//          64-bit DMA words. The OpenEye accelerator receives layer config
//          through a DMA interface, with parameters packed into multiple
//          64-bit transmissions to minimize transfer overhead.
//
// STRUCTURE: PARAMETER_POS_T_N
//   T = Transmission number (0-3)
//   N = Parameter index within that transmission
//   Value = Starting bit position in the 64-bit DMA word
//
// USAGE EXAMPLE (from dma_storage.v):
//   stride_x_reg <= dma_data_i[PARAMETER_POS_0_1 + 2 : PARAMETER_POS_0_1];
//   Extracts a 3-bit stride_x value from bits [10:8] of transmission 0

parameter DMA_BITWIDTH = 64;       // Width of DMA data bus
parameter TRANSMISSIONS = 4;       // Number of DMA transmissions per layer

// ========================================================================
// Transmission 0: Core layer execution parameters
// ========================================================================
// Contains fundamental execution control: strides, skips, kernel info
parameter PARAMETER_POS_0_0 = 0;                           // wght_cycles_reg (8 bits)
parameter PARAMETER_POS_0_1 = PARAMETER_POS_0_0 + 8;       // stride_x_reg (3 bits)
parameter PARAMETER_POS_0_2 = PARAMETER_POS_0_1 + 3;       // stride_y_reg (3 bits)
parameter PARAMETER_POS_0_3 = PARAMETER_POS_0_2 + 3;       // skipIact_reg (1 bit)
parameter PARAMETER_POS_0_4 = PARAMETER_POS_0_3 + 1;       // skipWght_reg (1 bit)
parameter PARAMETER_POS_0_5 = PARAMETER_POS_0_4 + 1;       // skipPsum_reg (1 bit)
parameter PARAMETER_POS_0_6 = PARAMETER_POS_0_5 + 1;       // psum_delay_reg (4 bits)
parameter PARAMETER_POS_0_7 = PARAMETER_POS_0_6 + 4;       // kernel_per_pe_cluster_reg (4 bits)
parameter PARAMETER_POS_0_8 = PARAMETER_POS_0_7 + 4;       // kernel_size (4 bits)
parameter PARAMETER_POS_0_9 = PARAMETER_POS_0_8 + 4;       // x_lines_reg (8 bits)
parameter PARAMETER_POS_0_10 = PARAMETER_POS_0_9 + 8;      // needed_wght_cycles_reg (8 bits)
parameter PARAMETER_POS_0_11 = PARAMETER_POS_0_10 + 8;     // needed_cycles_reg (18 bits)
// Total: 8+3+3+1+1+1+4+4+4+8+8+18 = 63 bits (fits in 64-bit word)

// ========================================================================
// Transmission 1: Input activation (iact) configuration
// ========================================================================
// Contains input feature map dimensions and buffering parameters
parameter PARAMETER_POS_1_0 = 0;                           // iact_converter_buffer_addr_max_cycles (8 bits)
parameter PARAMETER_POS_1_1 = PARAMETER_POS_1_0 + 8;       // iact_channels_per_pe (8 bits)
parameter PARAMETER_POS_1_2 = PARAMETER_POS_1_1 + 8;       // fc_size_reg (12 bits)
parameter PARAMETER_POS_1_3 = PARAMETER_POS_1_2 + 12;      // iact_size_x (8 bits)
parameter PARAMETER_POS_1_4 = PARAMETER_POS_1_3 + 8;       // iact_size_y (8 bits)
parameter PARAMETER_POS_1_5 = PARAMETER_POS_1_4 + 8;       // iact_needed_cycles (11 bits)
parameter PARAMETER_POS_1_6 = PARAMETER_POS_1_5 + 11;      // kernels_per_calc (5 bits)
parameter PARAMETER_POS_1_7 = PARAMETER_POS_1_6 + 5;       // y_lines_per_calc (4 bits)
// Total: 8+8+12+8+8+11+5+4 = 64 bits (exactly fills 64-bit word)

// ========================================================================
// Transmission 2: Output and layer type configuration
// ========================================================================
// Contains output parameters and layer type flags (pooling, FC, etc.)
parameter PARAMETER_POS_2_0 = 0;                           // output_cycles (8 bits)
parameter PARAMETER_POS_2_1 = PARAMETER_POS_2_0 + 8;       // store_in_psum (1 bit)
parameter PARAMETER_POS_2_2 = PARAMETER_POS_2_1 + 1;       // max_pooling (1 bit)
parameter PARAMETER_POS_2_3 = PARAMETER_POS_2_2 + 1;       // fully_connected_layer (1 bit)
parameter PARAMETER_POS_2_4 = PARAMETER_POS_2_3 + 1;       // choose_iact_buffer_output (1 bit)
parameter PARAMETER_POS_2_5 = PARAMETER_POS_2_4 + 1;       // choose_iact_buffer_input (1 bit)
parameter PARAMETER_POS_2_6 = PARAMETER_POS_2_5 + 1;       // iact_channels_per_pe_next_layer (4 bits)
parameter PARAMETER_POS_2_7 = PARAMETER_POS_2_6 + 4;       // needed_psum_storage_cycles_reg (8 bits)
parameter PARAMETER_POS_2_8 = PARAMETER_POS_2_7 + 8;       // iact_channel_max_cycles (8 bits)
parameter PARAMETER_POS_2_9 = PARAMETER_POS_2_8 + 8;       // input_activations_reg (5 bits)
parameter PARAMETER_POS_2_10 = PARAMETER_POS_2_9 + 5;      // filters_reg (6 bits)
parameter PARAMETER_POS_2_11 = PARAMETER_POS_2_10 + 6;     // needed_x_cls_reg (2 bits)
parameter PARAMETER_POS_2_12 = PARAMETER_POS_2_11 + 2;     // needed_y_cls_reg (4 bits)
parameter PARAMETER_POS_2_13 = PARAMETER_POS_2_12 + 4;     // needed_iact_cycles_reg (4 bits)
parameter PARAMETER_POS_2_14 = PARAMETER_POS_2_13 + 4;     // wght_addr_len_reg (5 bits)
parameter PARAMETER_POS_2_15 = PARAMETER_POS_2_14 + 5;     // iact_addr_len_reg (4 bits)
parameter PARAMETER_POS_2_16 = PARAMETER_POS_2_15 + 4;     // send_data_out (1 bit)
// Total: 8+1+1+1+1+1+4+8+8+5+6+2+4+4+5+4+1 = 64 bits

// ========================================================================
// Transmission 3: Additional buffer configuration
// ========================================================================
// Contains remaining buffer and accumulation parameters
parameter PARAMETER_POS_3_0 = 0;                           // needed_iact_buffer_words_reg (13 bits)
parameter PARAMETER_POS_3_1 = PARAMETER_POS_3_0 + 13;      // add_up_reg (2 bits)
// Total: 13+2 = 15 bits (partially fills 64-bit word)

`endif // REGMAP_PARAMS_VH
'''

regmap_params_vh_file = os.path.join(output_dir, 'regmap_params.vh')
with open(regmap_params_vh_file, 'w') as f:
    f.write(regmap_params_vh_content)

print(f"✅ Created regmap_params.vh: DMA register map")
print(f"   - DMA width: 64 bits")
print(f"   - Transmissions: 4 (one per layer configuration)")
print(f"   - Purpose: Pack ~40 layer config parameters into 4×64-bit DMA words")

# ============================================================================
# 3. COCOTB TESTBENCH
# ============================================================================
# Now create the cocotb testbench that uses these configurations

cocotb_testbench = '''#!/usr/bin/env python3
# This file is part of the OpenEye project.
# All rights reserved. © Fachhochschule Dortmund - University of Applied Sciences and Arts.
# SPDX-License-Identifier: SHL-2.1

"""Cocotb testbench for MNIST PyTorch model simulation"""

import sys
import os

# Add OpenEye to path
directory = os.path.abspath(os.path.join(os.path.dirname(os.path.realpath(__file__)), os.pardir))
sys.path.extend([directory, os.path.dirname(os.path.realpath(__file__))])

import cocotb
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge, Timer
import numpy as np
import logging

# Import OpenEye modules
import open_eye.test_utils_main as ptu
import open_eye.rtl_test_utils as rtl_test_utils
import open_eye.timing_parameters as tp
import open_eye.generic_test_utils as gtu
import open_eye.DRAM as DRAM
import open_eye.time_stamper as time_stamper
import open_eye.open_eye_parameters as oep
import open_eye.layer_parameters as lp
import open_eye.simple_layer_operations as slo
import open_eye.layer_execution_state as les
import open_eye.stream_dicts as strdic
from open_eye.model_loader import load_model
from open_eye.layer_adapter import adapt_layers_for_keras

logger = logging.getLogger("cocotb")
logger.setLevel(logging.INFO)

# Get environment variables
os.environ.setdefault("CLOCK_LEN", "10")
os.environ.setdefault("CLOCK_UNIT", "ns")
os.environ.setdefault("CLOCK_DELAY_INPUT", "100")
os.environ.setdefault("CLOCK_DELAY_UNIT_INPUT", "ps")
os.environ.setdefault("CLOCK_DELAY_OUTPUT", "100")
os.environ.setdefault("CLOCK_DELAY_UNIT_OUTPUT", "ps")

@cocotb.test()
async def mnist_pytorch_test(dut):
    """Test the OpenEye accelerator with MNIST PyTorch model"""
    
    logger.info("="*70)
    logger.info("MNIST PyTorch Model Test on OpenEye Accelerator")
    logger.info("="*70)
    
    # Get model path and test data directory from environment
    model_path = os.getenv("MODEL_PATH", "mnist_model_quantized.onnx")
    test_data_dir = os.getenv("TEST_DATA_DIR", "test_data")
    
    # Timing parameters
    clk_cycle = int(os.environ["CLOCK_LEN"])
    clk_cycle_unit = os.environ["CLOCK_UNIT"]
    clk_delay_in = int(os.environ["CLOCK_DELAY_INPUT"])
    clk_delay_unit_in = os.environ["CLOCK_DELAY_UNIT_INPUT"]
    clk_delay_out = int(os.environ["CLOCK_DELAY_OUTPUT"])
    clk_delay_unit_out = os.environ["CLOCK_DELAY_UNIT_OUTPUT"]
    
    time_printer = time_stamper.time_stamper()
    ptp = tp.PortTimingParameters()
    ptp.initiate_params(clk_cycle, clk_cycle_unit, clk_delay_in, 
                       clk_delay_unit_in, clk_delay_out, clk_delay_unit_out)
    
    # Load the model
    logger.info(f"Loading model from {model_path}...")
    model = load_model(model_path)
    logger.info(f"Model loaded with {len(model.layers)} layers")
    
    # Adapt layers
    adapted_layers = adapt_layers_for_keras(model.layers)
    model.layers = adapted_layers
    
    # Create DRAM contents
    logger.info("Initializing DRAM...")
    dram = DRAM.DRAMContents(model)
    time_printer.timestamp("Initialized DRAM", logger)
    
    sparse_iacts = 0
    sparse_wghts = 0
    dram.write_initial_data_to_dram(model, sparse_iacts, sparse_wghts)
    time_printer.timestamp("DRAM initialized with model data", logger)
    
    # Get OpenEye parameters
    serial = False
    openeye_parameter = oep.get_oep(serial)
    time_printer.timestamp("OpenEye parameters set", logger)
    
    # Start the clock
    clk = Clock(dut.clk_i, ptp.clk_cycle, units=ptp.clk_cycle_unit)
    cocotb.start_soon(clk.start())
    logger.info(f"Clock started: {ptp.clk_cycle} {ptp.clk_cycle_unit}")
    
    # Reset the DUT
    await cocotb.start_soon(rtl_test_utils.reset_all_signals(ptp, dut, openeye_parameter.SERIAL))
    time_printer.timestamp("All signals reset", logger)
    
    # Load test data
    test_labels = np.load(os.path.join(test_data_dir, 'test_labels.npy'))
    quantized_preds = np.load(os.path.join(test_data_dir, 'quantized_predictions.npy'))
    
    logger.info(f"Loaded {len(test_labels)} test samples")
    logger.info(f"Expected predictions: {quantized_preds}")
    
    # Process each layer
    layer_es = les.LayerExecutionState()
    max_layers = len(model.layers)
    
    for layer_number, layer in enumerate(model.layers):
        logger.info(f"\\nProcessing Layer {layer_number}: {layer.__class__.__name__}")
        
        if "Pooling" in str(layer):
            slo.pool(dram, layer, layer_number)
        elif "Flat" in str(layer):
            slo.flat(dram, layer, layer_number)
        else:
            # Create layer parameters
            layer_parameters = lp.LayerParameters(layer, openeye_parameter, 
                                                 layer_number, max_layers)
            time_printer.timestamp(f"Layer {layer_number} parameters created", logger)
            
            # Collect expected results
            calculated_results = ptu.collect_results(layer, layer_number, 
                                                     layer_parameters, dram, 
                                                     openeye_parameter.SERIAL)
            output_order = ptu.make_ref(openeye_parameter, layer_parameters, 
                                       layer, layer_number, dram, calculated_results)
            
            # Create streams
            dram_layer_content = [dram.fmap[layer_number], 
                                 dram.weights[layer_number], 
                                 dram.bias[layer_number]]
            stream = ptu.write_stream(openeye_parameter, layer_parameters, 
                                     layer, dram_layer_content, 
                                     sparse_iacts, sparse_wghts)
            
            time_printer.timestamp(f"Layer {layer_number} streams created", logger)
            
            # Execute layer on hardware
            for layer_repetition in range(layer_parameters.needed_total_transmissions):
                logger.info(f"  Executing transmission {layer_repetition+1}/"
                          f"{layer_parameters.needed_total_transmissions}")
                
                # Send stream to hardware
                status_thread = cocotb.start_soon(
                    rtl_test_utils.send_stream(ptp, dut, stream[layer_repetition], 
                                              openeye_parameter, layer_parameters, 
                                              layer_repetition)
                )
                await status_thread
                
                # Receive and check results if not skipping psum
                if stream[layer_repetition][strdic.stream_parallel_dict["status"]][
                    strdic.status_dict["skipPsum"]] != 1:
                    output_thread = cocotb.start_soon(
                        rtl_test_utils.receive_stream(ptp, dut, stream[layer_repetition], 
                                                     openeye_parameter, layer_parameters, 
                                                     layer_repetition)
                    )
                    await output_thread
            
            # Verify layer results
            assert ptu.compare_dram_with_ref(layer, calculated_results, 
                                            dram.fmap[1 + layer_number]), \\
                f"Layer {layer_number} output mismatch!"
            
            logger.info(f"✅ Layer {layer_number} completed successfully")
        
        # Apply batch normalization if present
        slo.batchnorm_output(layer, 512, layer_number, dram)
    
    logger.info("\\n" + "="*70)
    logger.info("MNIST PyTorch Test Completed Successfully!")
    logger.info("="*70)
    
    # Final assertion
    assert dut.rst_ni.value == 1, "Reset signal should be high!"
'''

cocotb_tb_file = os.path.join(output_dir, 'mnist_pytorch_tb.py')
with open(cocotb_tb_file, 'w') as f:
    f.write(cocotb_testbench)

print(f"✅ Created cocotb testbench: {cocotb_tb_file}")

print("\n" + "="*70)
print("REGISTER MAP EXPLANATION")
print("="*70)
print("""
The regmap_params.vh file defines how layer configuration is sent to hardware:

1. PROBLEM: Each layer needs ~40 configuration parameters (stride, kernel size,
   input dimensions, etc.) to execute correctly on the hardware.

2. SOLUTION: Pack these parameters into 64-bit DMA words and transmit them
   efficiently through the DMA interface.

3. STRUCTURE: Parameters are grouped into 4 transmissions:
   - Transmission 0: Core execution (strides, kernel info, cycles)
   - Transmission 1: Input activation config (dimensions, buffering)
   - Transmission 2: Output config (layer type flags, output params)
   - Transmission 3: Additional buffers (remaining parameters)

4. HOW IT WORKS: 
   - Python side: test_utils_main.py packs parameters into 64-bit words
   - DMA transfer: 4×64-bit words sent to hardware per layer
   - Hardware side: dma_storage.v unpacks using PARAMETER_POS_X_Y defines
   - Verilog extracts each field: data[POS+width-1 : POS]

This compact encoding minimizes DMA bandwidth while configuring complex layers!
""")


# Step 13: Run Cocotb Simulation (Optional)

The cell below will attempt to run the cocotb simulation directly from the notebook. This requires:
- Icarus Verilog (iverilog) or another Verilog simulator
- The OpenEye HDL sources  
- Cocotb and cocotb-test Python packages

The simulation will run as a subprocess and display real-time output including errors and progress.

In [ ]:
"""Run cocotb simulation directly in the notebook"""
import sys
import shutil
from datetime import datetime
import os

# Fix for running cocotb in Jupyter (which already has an event loop)
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("✅ Applied nest_asyncio patch for Jupyter compatibility")
except ImportError:
    print("⚠️  Installing nest_asyncio for Jupyter compatibility...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio", "-q"])
    import nest_asyncio
    nest_asyncio.apply()
    print("✅ Installed and applied nest_asyncio")

def run_cocotb_simulation_direct(output_dir):
    """
    Run the cocotb simulation directly using cocotb_test API
    
    Args:
        output_dir: Directory containing the model and test data
    
    Returns:
        bool: True if successful, False otherwise
    """
    print("\n" + "="*70)
    print("COCOTB SIMULATION")
    print("="*70)
    
    # Check if iverilog is installed
    iverilog_path = shutil.which('iverilog')
    if not iverilog_path:
        print("❌ Icarus Verilog (iverilog) not found!")
        print("\nTo install:")
        print("  macOS:   brew install icarus-verilog")
        print("  Ubuntu:  sudo apt-get install iverilog")
        print("  Windows: Download from http://bleyer.org/icarus/")
        return False
    
    print(f"✅ Found iverilog: {iverilog_path}")
    
    # Check if cocotb is installed
    try:
        import cocotb
        import cocotb_test.simulator
        print(f"✅ Found cocotb version: {cocotb.__version__}")
    except ImportError as e:
        print(f"❌ Cocotb not installed: {e}")
        print("\nTo install:")
        print("  pip install cocotb cocotb-test")
        return False
    
    # Check if HDL sources exist
    hdl_dir = os.path.abspath(os.path.join(os.pardir, os.pardir, "hdl"))
    if not os.path.exists(hdl_dir):
        print(f"❌ HDL directory not found: {hdl_dir}")
        print("\nMake sure you're running this from the OpenEye repository.")
        return False
    
    print(f"✅ Found HDL directory: {hdl_dir}")
    
    # Check if parameters.vh exists
    parameters_vh = os.path.abspath(os.path.join(output_dir, 'parameters.vh'))
    if not os.path.exists(parameters_vh):
        print(f"❌ parameters.vh not found: {parameters_vh}")
        print("   Make sure you ran the previous cell to create it.")
        return False
    
    print(f"✅ Found parameters.vh: {parameters_vh}")
    
    # Import OpenEye test utils
    sys.path.insert(0, os.path.abspath(os.path.join(os.pardir, os.pardir, "src")))
    sys.path.insert(0, os.path.abspath(os.path.join(os.pardir, os.pardir, "test")))
    
    try:
        import open_eye.test_utils_main as test_utils_main
    except ImportError as e:
        print(f"❌ Could not import OpenEye modules: {e}")
        return False
    
    print("✅ OpenEye modules loaded")
    
    # Setup paths
    model_path = os.path.abspath(os.path.join(output_dir, 'mnist_model_quantized.onnx'))
    test_data_dir = os.path.abspath(os.path.join(output_dir, 'test_data'))
    
    print(f"\n📦 Model: {model_path}")
    print(f"📁 Test data: {test_data_dir}")
    
    # Get Verilog sources
    verilog_sources = test_utils_main.get_verilog_sources(hdl_dir)
    print(f"✅ Found {len(verilog_sources)} Verilog source files")
    
    # Clock configuration
    clk_cycle = 20
    clk_cycle_unit = "ns"
    clk_delay_in = 100
    clk_delay_unit_in = "ps"
    clk_delay_out = 100
    clk_delay_unit_out = "ps"
    
    # Build directory
    sim_build_dir = os.path.join(output_dir, '.temp', 'mnist_pytorch_test')
    
    print("\n" + "="*70)
    print(f"Starting simulation at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print("\n⏳ This may take several minutes...")
    print("   (Compiling Verilog, running simulation, validating results)\n")
    
    try:
        # Run cocotb simulation directly
        # nest_asyncio allows this to work in Jupyter's existing event loop
        results = cocotb_test.simulator.run(
            python_search=[output_dir],
            verilog_sources=verilog_sources,
            toplevel='OpenEye_Parallel',
            module='mnist_pytorch_tb',
            sim_build=sim_build_dir,
            testcase='mnist_pytorch_test',
            force_compile=True,
            simulator="icarus",
            # Add include path for parameters.vh
            includes=[output_dir],
            # Compile arguments to add include directory
            compile_args=[f"-I{output_dir}"],
            extra_env={
                "CLOCK_LEN": str(clk_cycle),
                "CLOCK_UNIT": clk_cycle_unit,
                "CLOCK_DELAY_INPUT": str(clk_delay_in),
                "CLOCK_DELAY_UNIT_INPUT": clk_delay_unit_in,
                "CLOCK_DELAY_OUTPUT": str(clk_delay_out),
                "CLOCK_DELAY_UNIT_OUTPUT": clk_delay_unit_out,
                "MODEL_PATH": model_path,
                "TEST_DATA_DIR": test_data_dir
            }
        )
        
        print("\n" + "="*70)
        print("✅ SIMULATION COMPLETED SUCCESSFULLY!")
        print("="*70)
        print(f"Finished at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        return True
        
    except Exception as e:
        print("\n" + "="*70)
        print("❌ SIMULATION FAILED")
        print("="*70)
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
        return False

# Run the simulation directly
print("Running cocotb simulation directly in the notebook...\n")
success = run_cocotb_simulation_direct(output_dir)

if success:
    print("\n🎉 Simulation completed successfully!")
    print("\n💡 Results:")
    print("  ✓ Model layers simulated on OpenEye hardware")
    print("  ✓ Hardware outputs validated against golden reference")
    print("  ✓ All layer computations verified")
    print(f"\n📁 Simulation artifacts saved to: {output_dir}/.temp/")
else:
    print("\n⚠️  Simulation could not run or failed.")
    print("\n💡 Common issues:")
    print("  - Missing dependencies (iverilog, cocotb)")
    print("  - HDL sources not found")
    print("  - Model or test data files missing")
    print("  - parameters.vh not created (run previous cells)")
    print("\nYou can also run manually:")
    print(f"  cd {output_dir}")
    print(f"  python test_mnist_pytorch.py")


In [ ]:
"""Summary of generated files"""

print("="*70)
print("Cocotb Simulation Setup Complete!")
print("="*70)
print(f"\nAll files saved to: {output_dir}/")
print("\nGenerated files:")
print(f"  1. {os.path.basename(cocotb_test_file)} - Main test script")
print(f"  2. {os.path.basename(cocotb_tb_file)} - Cocotb testbench")
print(f"  3. parameters.vh - Hardware parameters for Verilog")
print(f"  4. regmap_params.vh - Register map parameters for Verilog")
print(f"  5. test_data/ - Test data directory")
print(f"  6. {os.path.basename(onnx_model_path)} - Quantized model (ONNX format)")

print("\n" + "="*70)
print("Directory Structure:")
print("="*70)
print(f"{output_dir}/")
print(f"├── mnist_model_quantized.onnx")
print(f"├── parameters.vh              ← Verilog hardware parameters")
print(f"├── regmap_params.vh           ← Verilog register map parameters")
print(f"├── test_mnist_pytorch.py")
print(f"├── mnist_pytorch_tb.py")
print(f"└── test_data/")
print(f"    ├── test_image_0.npy ... test_image_9.npy")
print(f"    ├── test_labels.npy")
print(f"    └── quantized_predictions.npy")

print("\n✅ All files ready for cocotb simulation!")
print("\n💡 Next: Run Step 14 to execute the simulation")


## Summary

This notebook demonstrated the **complete end-to-end workflow** for deploying a PyTorch MNIST model on the OpenEye neural network accelerator:

### Training & Quantization
✅ Trained a CNN model on MNIST dataset  
✅ Quantized the model to INT8 using PyTorch's quantization tools  
✅ Validated quantized model accuracy  

### Model Export & Loading
✅ Exported quantized model to ONNX format  
✅ Loaded model with OpenEye's unified model loader  
✅ Adapted layers for hardware compatibility  

### Cocotb Simulation Setup
✅ Created cocotb test script for hardware simulation  
✅ Created cocotb testbench with OpenEye integration  
✅ Prepared test data for validation  
✅ **Organized all files in `mnist_pytorch_cocotb/` directory**  
✅ **Added simulation runner with real-time output**  

### Key Features Demonstrated
- **Multi-framework support**: PyTorch → ONNX → OpenEye
- **Quantization-aware workflow**: INT8 quantization with dynamic batch sizes
- **Hardware simulation**: Full cocotb integration with OpenEye RTL
- **Layer-by-layer execution**: Conv2D, MaxPooling, Dense layers on hardware
- **Result validation**: Automatic comparison with golden reference
- **Real-time monitoring**: Simulation runs directly from notebook with live output

### Architecture
```
Input (28×28×1)
    ↓
Conv2D + ReLU (quantized)
    ↓
MaxPooling2D
    ↓
Conv2D + ReLU (quantized)
    ↓
MaxPooling2D
    ↓
Flatten
    ↓
Dense (quantized)
    ↓
Output (10 classes)
```

### Performance Benefits
- **Model size**: ~4x reduction through INT8 quantization
- **Inference speed**: Faster execution on specialized hardware
- **Accuracy**: Minimal loss (<1-2%) with quantization
- **Hardware efficiency**: Optimized for OpenEye PE clusters

### Generated Files (in `mnist_pytorch_cocotb/`)
```
mnist_pytorch_cocotb/
├── mnist_model_quantized.onnx    - Quantized model
├── test_mnist_pytorch.py         - Main test runner
├── mnist_pytorch_tb.py           - Cocotb testbench
└── test_data/                    - Test samples and expected results
    ├── test_image_*.npy
    ├── test_labels.npy
    └── quantized_predictions.npy
```

### Running the Simulation

**Option 1: From the notebook (Step 14)**
- Run the simulation cell to execute cocotb directly
- Real-time output shows progress and errors
- Automatic dependency checking

**Option 2: From command line**
```bash
cd mnist_pytorch_cocotb
python test_mnist_pytorch.py
```

**Option 3: With pytest**
```bash
cd mnist_pytorch_cocotb
pytest test_mnist_pytorch.py -v
```

### Related Notebooks
- [simple_cnn_demo.ipynb](simple_cnn_demo.ipynb) - Simplified CNN workflow
- [mnist_tensorflow.ipynb](mnist_tensorflow.ipynb) - TensorFlow version
- [unified_model_loader_demo.ipynb](unified_model_loader_demo.ipynb) - Multi-framework examples